# SheridanV2 — Paperspace ComfyUI setup

This notebook installs the ComfyUI server alongside the existing SheridanV2 checkout. It deliberately preserves the current PyTorch/CUDA installation. It does **not** download SDXL checkpoints or InstantID weights yet; those come after the server passes its health check.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

PROJECT = Path('/notebooks/SheridanV2')
COMFY_DIR = Path('/notebooks/ComfyUI')
assert PROJECT.exists(), f'SheridanV2 checkout not found at {PROJECT}. Update PROJECT if your clone is elsewhere.'
import torch
print('Project:', PROJECT)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('ComfyUI directory already exists:', COMFY_DIR.exists())

In [ ]:
# Clone ComfyUI if it is not already present.
if not COMFY_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print('Keeping existing ComfyUI checkout; no overwrite or reset performed.')

# Install ComfyUI's Python requirements without changing the working PyTorch stack.
req_file = COMFY_DIR / 'requirements.txt'
requirements = []
for raw in req_file.read_text().splitlines():
    line = raw.strip()
    if not line or line.startswith('#'):
        continue
    package = line.split('=')[0].split('<')[0].split('>')[0].split('!')[0].strip().lower()
    if package in {'torch', 'torchvision', 'torchaudio'}:
        print('Preserving installed package; skipping:', line)
        continue
    requirements.append(line)
filtered = COMFY_DIR / 'requirements-paperspace-filtered.txt'
filtered.write_text('\n'.join(requirements) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(filtered)], check=True)
print('ComfyUI files and non-PyTorch requirements are ready.')

In [ ]:
# Start ComfyUI in the background if its API is not already responding.
import time, urllib.request, urllib.error, subprocess
def comfy_healthy():
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3) as response:
            return response.status == 200
    except Exception:
        return False

if comfy_healthy():
    print('ComfyUI is already responding at http://127.0.0.1:8188')
else:
    log_path = Path('/notebooks/comfyui.log')
    log_handle = open(log_path, 'a', buffering=1)
    process = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
        cwd=str(COMFY_DIR), stdout=log_handle, stderr=subprocess.STDOUT,
        start_new_session=True
    )
    print('Started ComfyUI process:', process.pid)
    for attempt in range(30):
        time.sleep(2)
        if comfy_healthy():
            print('HEALTH CHECK PASSED: ComfyUI API is responding on port 8188.')
            break
    else:
        print('ComfyUI has not responded yet. Recent log output:')
        print(log_path.read_text(errors='replace')[-5000:])
        raise RuntimeError('ComfyUI did not become healthy. Do not install models until this is resolved.')

In [ ]:
# Confirm that ComfyUI sees the same GPU and expose its node inventory endpoint.
import json, urllib.request
with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=10) as response:
    stats = json.load(response)
print(json.dumps(stats, indent=2)[:6000])
print('\nNext checkpoint: confirm GPU visibility and ComfyUI health before adding InstantID custom nodes or model weights.')